# 09 — State-conditioned verifier V2 architecture sweep

This notebook fixes best-epoch restoration, compares multiplicative, FiLM,
cross-attention, action-only, and deranged-action controls, and registers one
checkpoint bundle. It never loads confirmatory candidate outcomes.

## 1. Setup

In [ ]:
import os, subprocess, sys
try:
    from google.colab import userdata
    for key in ("SUPABASE_URL", "SUPABASE_SERVICE_KEY", "HF_TOKEN", "WANDB_API_KEY"):
        value = userdata.get(key)
        if value: os.environ[key] = value
    repo_dir = "/content/cs159-sp26"
    gh_pat = userdata.get("GH_PAT")
    repo_url = f"https://{gh_pat}@github.com/ArjunS07/cs159-sp26.git"
    if not os.path.isdir(os.path.join(repo_dir, ".git")):
        subprocess.run(["git", "clone", "--branch", "main", repo_url, repo_dir], check=True)
    else:
        subprocess.run(["git", "-C", repo_dir, "pull", "--ff-only", "origin", "main"], check=True)
except ImportError:
    repo_dir = os.path.abspath("..") if os.path.basename(os.getcwd()) == "pnp-vla" else os.getcwd()
package_dir = os.path.join(repo_dir, "pnp-vla")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e",
                package_dir + "[sim,analysis]"], check=True)
if package_dir not in sys.path: sys.path.insert(0, package_dir)
import pnp
print("Loaded pnp from:", pnp.__file__)

## 2. Configuration and data

In [ ]:
from dataclasses import replace
from pathlib import Path
import copy, json, pickle
import numpy as np
import pandas as pd
import torch
import wandb
from tqdm.auto import tqdm
from pnp.store import SupabaseStore
from pnp.verifier import *

DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
OUTPUT=Path(package_dir)/"analysis_outputs"/"verifier_v2"; OUTPUT.mkdir(parents=True, exist_ok=True)
store=SupabaseStore()
HISTORICAL=("libero-hybrid-schedules-k3-v1", "libero-pro-canonical-core-k3-v1")
PRIMARY=("verifier-clean-pairs-v3", "verifier-clean-pairs-v4-dev",
         "verifier-clean-pairs-v4-test", "verifier-online-selection-v1",
         "verifier-v2-pro-development")
CONFIRMATORY="verifier-v2-pro-confirmatory"
SEEDS=(42,43,44); N_FOLDS=4; PREFIX_LENGTH=10

historical=load_clean_chunk_examples(
    store, HISTORICAL, progress=tqdm, cache_dir=OUTPUT/"historical_cache")
existing_development=load_candidate_examples(
    store, PRIMARY[:-1], cache_dir=OUTPUT/"candidate_cache_existing")
new_development=load_candidate_examples(
    store, PRIMARY[-1], cache_dir=OUTPUT/"candidate_cache_v2")
new_audit=validate_candidate_groups(new_development,expected_candidates=12)
assert new_audit["groups"] >= 220, new_audit
development=existing_development+new_development
audit=validate_candidate_groups(development)
assert audit["discordant_groups"] >= 100, audit
# Seal IDs only. Do not query verifier_candidates for this experiment here.
sealed=(store.client.table("verifier_candidate_groups").select(
        "candidate_group_id,benchmark,suite,task_idx,episode_idx")
        .eq("experiment", CONFIRMATORY).execute().data or [])
assert len(sealed) >= 120, len(sealed)
sealed_identities={(row["benchmark"],row["suite"],int(row["task_idx"]),
                    int(row["episode_idx"])) for row in sealed}
historical=exclude_episode_identities(historical,sealed_identities)
folds=candidate_cv_splits(development, [e.rollout_id for e in development], N_FOLDS, 42)
print({"historical_chunks":len(historical), "development":audit,
       "new_PRO_development":new_audit,
       "sealed_confirmatory_groups":len(sealed), "device":str(DEVICE)})

## 3. Fixed sweep runner

In [ ]:
def historical_fold(candidate_validation, fold_index):
    protected=candidate_episode_identities(candidate_validation)
    clean=exclude_episode_identities(historical, protected)
    split=known_task_split(clean, seed=420+fold_index)
    val=select_examples(clean, split["val"]); val_ids=set(split["val"])
    return [e for e in clean if e.rollout_id not in val_ids], val

VALUE_CACHE={}
def pretrained_state(fold_index,seed,dropout):
    key=(fold_index,seed,dropout)
    if key not in VALUE_CACHE:
        val=select_examples(development,folds[fold_index]["val"])
        value_train,value_val=historical_fold(val,fold_index)
        cfg=AdvantageTrainConfig(seed=seed,prefix_length=PREFIX_LENGTH,
            value_epochs=60,patience=7,weight_decay=1e-2)
        base=CompactAdvantageVerifier(action_width=64,dropout=dropout)
        base,meta=pretrain_value(base,value_train,value_val,DEVICE,config=cfg)
        VALUE_CACHE[key]=(copy.deepcopy(base.state_dict()),meta)
    return VALUE_CACHE[key]

def fit_one(spec, fold_index, seed, max_epochs, patience):
    fold=folds[fold_index]
    train=select_examples(development, fold["train"])
    val=select_examples(development, fold["val"])
    config=AdvantageTrainConfig(
        seed=seed,prefix_length=PREFIX_LENGTH,value_epochs=60,rank_epochs=max_epochs,
        patience=patience,rank_lr=spec["rank_lr"],weight_decay=1e-2,
        context_lr_multiplier=.1,zero_context=spec.get("action_only",False))
    model=CompactAdvantageVerifier(action_width=64,dropout=spec["dropout"],
                                   conditioning=spec["architecture"])
    state,value_meta=pretrained_state(fold_index,seed,spec["dropout"])
    model.load_state_dict(state)
    if spec.get("shuffle"):
        train=shuffle_candidate_actions_within_group(train, seed)
    model, rank_meta=train_advantage(model,train,val,DEVICE,config=config)
    metrics,records=evaluate_candidate_ranker(
        model,val,DEVICE,config=config,n_bootstrap=1000,return_records=True)
    return model,config,{**value_meta,**rank_meta},metrics,records

ARCHITECTURES=("multiplicative","film","cross_attention")
STAGE1=[{"name":f"{a}-d{d}-lr{lr}","architecture":a,"dropout":d,"rank_lr":lr}
        for a in ARCHITECTURES for d in (.2,.4) for lr in (1e-4,3e-4)]
ACTION={"name":"action-only","architecture":"action_only","dropout":.2,
        "rank_lr":3e-4,"action_only":True}
SHUFFLED={"name":"shuffled-actions","architecture":"film","dropout":.2,
          "rank_lr":3e-4,"shuffle":True}
print({"stage1_configs":len(STAGE1),"folds":N_FOLDS})

## 4. Stage 1: all architectures, one seed

In [ ]:
stage1_rows=[]; stage1_records={}; history_rows=[]
for spec in STAGE1+[ACTION,SHUFFLED]:
    all_records=[]
    for fold_index in range(N_FOLDS):
        _,_,meta,metrics,records=fit_one(spec,fold_index,42,30,5)
        stage1_rows.append({"name":spec["name"],"fold":fold_index,
                            "ranking":metrics["group_macro_ranking_accuracy"],
                            "uplift":metrics["top1_uplift_default"],
                            "margin":metrics["mean_score_margin"],
                            "best_epoch":meta["best_rank_epoch"]})
        history_rows += [{**point,"stage":1,"name":spec["name"],
                          "fold":fold_index,"seed":42}
                         for point in meta["rank_history"]]
        all_records += records
    stage1_records[spec["name"]]=all_records
stage1=pd.DataFrame(stage1_rows)
summary=stage1.groupby("name").agg(ranking=("ranking","mean"),
                                    uplift=("uplift","mean"),best_epoch=("best_epoch","median"))
action_ranking=summary.loc[ACTION["name"],"ranking"]
shuffled_ranking=summary.loc[SHUFFLED["name"],"ranking"]
eligible=summary.loc[[s["name"] for s in STAGE1]].copy()
eligible["control_gap"]=eligible.ranking-max(action_ranking,shuffled_ranking)
shortlist=list(eligible.sort_values(["control_gap","ranking","uplift"],ascending=False).head(2).index)
assert len(shortlist)==2
stage1.to_csv(OUTPUT/"stage1_results.csv",index=False)
print(summary.sort_values("ranking",ascending=False)); print("shortlist",shortlist)

## 5. Stage 2: shortlisted configurations × three seeds

In [ ]:
spec_by_name={spec["name"]:spec for spec in STAGE1}
stage2_specs=[spec_by_name[name] for name in shortlist]+[ACTION,SHUFFLED]
stage2_rows=[]; pooled={spec["name"]:[] for spec in stage2_specs}
for spec in stage2_specs:
    name=spec["name"]
    for seed in SEEDS:
        for fold_index in range(N_FOLDS):
            _,_,meta,metrics,records=fit_one(spec,fold_index,seed,50,7)
            stage2_rows.append({"name":name,"seed":seed,"fold":fold_index,
                                "ranking":metrics["group_macro_ranking_accuracy"],
                                "uplift":metrics["top1_uplift_default"],
                                "margin":metrics["mean_score_margin"],
                                "best_epoch":meta["best_rank_epoch"]})
            history_rows += [{**point,"stage":2,"name":name,
                              "fold":fold_index,"seed":seed}
                             for point in meta["rank_history"]]
            pooled[name] += records
stage2=pd.DataFrame(stage2_rows)
stage2_summary=stage2.groupby("name").agg(
    ranking=("ranking","mean"),ranking_std=("ranking","std"),
    uplift=("uplift","mean"),best_epoch=("best_epoch","median"))
stage2_action=stage2_summary.loc[ACTION["name"],"ranking"]
stage2_shuffled=stage2_summary.loc[SHUFFLED["name"],"ranking"]
stage2_summary["control_gap"]=stage2_summary.ranking-max(stage2_action,stage2_shuffled)
selected=stage2_summary.loc[shortlist].sort_values(
    ["control_gap","ranking","uplift"],ascending=False).index[0]
stage2.to_csv(OUTPUT/"stage2_results.csv",index=False)
pd.DataFrame(history_rows).to_csv(OUTPUT/"training_curves.csv",index=False)
if os.getenv("WANDB_API_KEY"):
    run=wandb.init(project="pnp-state-conditioned-verifier-v2",name="architecture-sweep")
    for name,row in stage2_summary.iterrows():
        run.log({f"summary/{name}/ranking":row.ranking,
                 f"summary/{name}/uplift":row.uplift,
                 f"summary/{name}/control_gap":row.control_gap})
    artifact=wandb.Artifact("verifier-v2-sweep","evaluation")
    for filename in ("stage1_results.csv","stage2_results.csv","training_curves.csv"):
        artifact.add_file(str(OUTPUT/filename))
    run.log_artifact(artifact); run.finish()
print(stage2_summary); print("selected",selected)

## 6. Final refits and single checkpoint bundle

In [ ]:
selected_spec=spec_by_name[selected]
selected_records=aggregate_candidate_records(pooled[selected])
action_records=aggregate_candidate_records(pooled[ACTION["name"]])
shuffled_records=aggregate_candidate_records(pooled[SHUFFLED["name"]])
development_metrics=summarize_candidate_records(selected_records)
action_comparison=paired_candidate_comparison(
    selected_records,action_records,n_bootstrap=2000)
shuffled_comparison=paired_candidate_comparison(
    selected_records,shuffled_records,seed=43,n_bootstrap=2000)
development_gate=verifier_registration_eligibility(
    development_metrics,action_comparison,shuffled_comparison)
development_report={"metrics":development_metrics,"vs_action_only":action_comparison,
                    "vs_shuffled_actions":shuffled_comparison,"gate":development_gate}
epoch_by_name={name:max(1,int(rows.best_epoch.median())+1)
               for name,rows in stage2.groupby("name")}
final_epochs=epoch_by_name[selected]
split=known_task_split(historical,seed=159)
value_val=select_examples(historical,split["val"]); value_val_ids=set(split["val"])
value_train=[e for e in historical if e.rollout_id not in value_val_ids]

def final_fit(spec, train_examples, rank_epochs):
    cfg=AdvantageTrainConfig(seed=159,prefix_length=10,value_epochs=60,
        rank_epochs=rank_epochs,patience=100,rank_lr=spec["rank_lr"],weight_decay=1e-2,
        context_lr_multiplier=.1,zero_context=spec.get("action_only",False))
    model=CompactAdvantageVerifier(action_width=64,dropout=spec["dropout"],
                                   conditioning=spec["architecture"])
    model,_=pretrain_value(model,value_train,value_val,DEVICE,config=cfg)
    model,_=train_advantage(model,train_examples,[],DEVICE,config=cfg)
    return model,cfg

selected_model,selected_cfg=final_fit(selected_spec,development,epoch_by_name[selected])
action_model,action_cfg=final_fit(ACTION,development,epoch_by_name[ACTION["name"]])
shuffled_model,shuffled_cfg=final_fit(
    SHUFFLED,shuffle_candidate_actions_within_group(development,159),
    epoch_by_name[SHUFFLED["name"]])
bundle={"model":selected_model.state_dict(),"controls":{
    "action_only":{"model":action_model.state_dict(),"spec":ACTION},
    "shuffled_actions":{"model":shuffled_model.state_dict(),"spec":SHUFFLED}},
    "metadata":{"selected_spec":selected_spec,"final_rank_epochs":final_epochs,
                "parameter_count":sum(p.numel() for p in selected_model.parameters()),
                "development_report":development_report,
                "development_records":selected_records,
                "action_control_records":action_records,
                "shuffled_control_records":shuffled_records,
                "stage2_summary":stage2_summary.reset_index().to_dict("records")}}
import io
buffer=io.BytesIO(); torch.save(bundle,buffer)
verifier_id=new_verifier_id()
store.start_run("verifier_train","libero+libero_pro","state-conditioned-verifier-v2",
                config={"selected_spec":selected_spec,
                        "final_rank_epochs":final_epochs,
                        "development_gate":development_gate})
store.register_verifier(verifier_id,buffer.getvalue(),{
    "model_class":"CompactAdvantageVerifier","obs_dim":2048,"action_dim":7,
    "horizon":50,"prefix_length":10,"architecture":selected_spec["architecture"],
    "action_width":64,"dropout":selected_spec["dropout"]},
    {"development_stage2":stage2_summary.reset_index().to_dict("records"),
     "development_report":development_report},
    {"folds":folds,"sealed_confirmatory_group_ids":sorted(
        row["candidate_group_id"] for row in sealed)},dataset_hash=dataset_hash(development))
store.finish_run(n_rollouts=len(development))
(OUTPUT/"registered_verifier.json").write_text(json.dumps({
    "verifier_id":verifier_id,"selected":selected_spec,"final_epochs":final_epochs},indent=2))
print({"registered_verifier":verifier_id,"selected":selected_spec,
       "final_rank_epochs":final_epochs})